<a href="https://colab.research.google.com/github/gpreti/EE-629/blob/main/EX2/EX_dFC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ========================================
# DYNAMIC FUNCTIONAL CONNECTIVITY EXERCISE
# ========================================

# ===================== LIBRARIES =====================
import os
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.stats import zscore

# ============================
# Clone entire Github repository
# ============================

!git clone https://github.com/gpreti/EE-629.git

print("Repository cloned.") #everything exist locally now, path: "EE-629/EX1/"

##### to pull when adding new files into github
#%cd /content/EE-629
#!git pull

In [ ]:
!ls EE-629/EX1/DATA

In [ ]:
path_exercise='EE-629/EX2/'

# ===================== LOAD TIMECOURSES =====================
tcs_file = os.path.join(path_exercise, 'DATA', 'TCS.mat')
tcs = scipy.io.loadmat(tcs_file)
print("Variables in .mat file:", tcs.keys())

# Extract the main timecourse variable
TCSvar = tcs['X1']  # shape: (regions, timepoints, subjects)
print("TCS shape (regions x timepoints x subjects):", TCSvar.shape)

n_regions, n_timepoints, n_subjects = TCSvar.shape

In [ ]:
# ===================== FUNCTION TO COMPUTE SLIDING-WINDOW DFC =====================

def sliding_window_dfc(TCS, window_size, step, fisher_z=False):
    """
    Compute sliding window dynamic functional connectivity (Pearson correlation).

    Parameters
    ----------
    TCS : np.ndarray
        Timecourses matrix of shape (n_regions, n_timepoints)
    window_size : int
        Window length in TRs
    step : int
        Step size to move the window (in TRs)
    fisher_z : bool
        If True, apply Fisher z-transform to correlations

    Returns
    -------
    dFC : np.ndarray
        Dynamic FC array of shape (n_windows, n_regions, n_regions)
    win_starts : list
        Starting indices of each window
    """

    n_regions, n_timepoints = TCS.shape

    # Compute window start indices
    win_starts = list(range(0, n_timepoints - window_size + 1, step))
    n_windows = len(win_starts)

    # Preallocate output
    dFC = np.zeros((n_windows, n_regions, n_regions))

    for i, start in enumerate(win_starts):
        end = start + window_size

        # Extract windowed data
        window_data = TCS[:, start:end]

        # Compute correlation matrix
        corr_matrix = np.corrcoef(window_data)

        if fisher_z:
            # Avoid infinite values
            corr_matrix = np.clip(corr_matrix, -0.999999, 0.999999)
            corr_matrix = np.arctanh(corr_matrix)

        dFC[i] = corr_matrix

    return dFC, win_starts

In [1]:
#compute dFC: define parameters window size and step in TR!
window_size = 60
step = 5

dFC, starts = sliding_window_dfc(TCSvar, window_size, step)
n_windows = dFC.shape[0]


NameError: name 'sliding_window_dfc' is not defined

In [ ]:
#Plot timecourse of selected connections
connections = [(0, 1), (2, 5), (3, 10)]  # choose some pairs

plt.figure(figsize=(10, 5))

for (i, j) in connections:
    plt.plot(dFC[:, i, j], label=f'Conn {i}-{j}')

plt.xlabel("Window index")
plt.ylabel("Correlation")
plt.title("Dynamic FC of selected connections")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

#Plot few FC matrices from different windows to check variations

windows_to_plot = [0, n_windows//3, 2*n_windows//3, n_windows-1]

fig, axes = plt.subplots(1, len(windows_to_plot), figsize=(15, 4))

for ax, w in zip(axes, windows_to_plot):
    im = ax.imshow(dFC[w], vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_title(f'Window {w}')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6)
plt.suptitle("FC matrices across windows")
plt.tight_layout()
plt.show()

In [ ]:

#Create video of FC matrices

fig, ax = plt.subplots()
im = ax.imshow(dFC[0], vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar(im)
ax.set_title("Dynamic FC")

def update(frame):
    im.set_array(dFC[frame])
    ax.set_title(f"Window {frame}")
    return [im]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=n_windows,
    interval=200,   # ms between frames
    blit=True
)

# Save video (requires ffmpeg installed)
ani.save("dynamic_FC.mp4", writer="ffmpeg", dpi=200)

plt.show()

In [ ]:
#Which connections vary the most in your group?
